## 📑 Table of Contents

* [Why This Notebook Is Non-Negotiable](#why-this-notebook-is-non-negotiable)
  * [🎓 Junior to Senior: Concept Bridge](#junior-to-senior-concept-bridge)
* [1 · Pandas Mastery: The Operations You'll Use Daily](#1-pandas-mastery-the-operations-youll-use-daily)
* [2 · Visual EDA: Finding Problems Your Eyes Catch Faster Than Code](#2-visual-eda-finding-problems-your-eyes-catch-faster-than-code)
* [3 · Data Cleaning Pipeline](#3-data-cleaning-pipeline)
* [4 · Feature Engineering for ML](#4-feature-engineering-for-ml)
* [5 · Data Validation (Connects to NB07)](#5-data-validation-connects-to-nb07)
* [✅ Knowledge Check](#knowledge-check)
  * [Q1: Imputation strategy](#q1-imputation-strategy)
  * [Q2: Data leakage](#q2-data-leakage)
  * [Q3: When NOT to scale](#q3-when-not-to-scale)
* [🔨 Practical Practice](#practical-practice)
  * [Exercise 1: Missing Data Investigation](#exercise-1-missing-data-investigation)
  * [Exercise 2: Feature Interaction](#exercise-2-feature-interaction)
  * [Exercise 3: Full Pipeline](#exercise-3-full-pipeline)


### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Juniors treat EDA as "make some plots." Seniors use EDA to discover data quality issues BEFORE they corrupt a model. One missing-value strategy can shift model accuracy by 5%+. One leaky feature can make your model useless in production.

**Mental Model:** EDA is a medical exam for your data. You're checking vital signs (distributions), looking for symptoms (outliers, nulls), and diagnosing conditions (data leakage, class imbalance) before surgery (model training).

**Common Junior Pitfall:** Training on data with target leakage — a feature that"magically" contains information from the future. Example: including `account_closed_date` to predict churn. The model gets 99% accuracy in dev, 50% in production.

---

In [ ]:
!pip install -q pandas numpy matplotlib seaborn scikit-learn

In [ ]:
# Cell 1 — Create realistic messy dataset
import pandas as pd
import numpy as np

np.random.seed(42)
n = 1000

data = pd.DataFrame({
    'customer_id': range(1000, 1000+n),
    'age': np.random.normal(35, 12, n).astype(int),
    'income': np.random.lognormal(10.5, 0.8, n).round(2),
    'credit_score': np.random.normal(680, 80, n).astype(int),
    'account_age_months': np.random.exponential(36, n).astype(int),
    'num_products': np.random.poisson(2.5, n),
    'has_credit_card': np.random.choice([0, 1], n, p=[0.3, 0.7]),
    'is_active': np.random.choice([0, 1], n, p=[0.2, 0.8]),
    'country': np.random.choice(['US', 'UK', 'DE', 'FR', 'JP', 'BR'], n),
    'satisfaction_score': np.random.choice([1,2,3,4,5], n, p=[0.05,0.1,0.2,0.35,0.3]),
    'churned': np.random.choice([0, 1], n, p=[0.8, 0.2]),  # Target variable
})

# Inject REALISTIC data problems
# 1. Missing values (5-15% in various columns)
for col in ['income', 'credit_score', 'satisfaction_score']:
    mask = np.random.random(n) < np.random.uniform(0.05, 0.15)
    data.loc[mask, col] = np.nan

# 2. Outliers (data entry errors)
data.loc[42, 'age'] = 350       # Impossible age
data.loc[99, 'income'] = -5000  # Negative income
data.loc[150, 'credit_score'] = 9999  # Invalid score

# 3. Duplicates
data = pd.concat([data, data.iloc[:5]], ignore_index=True)  # 5 duplicate rows

print(f'Dataset: {data.shape[0]} rows × {data.shape[1]} columns')
print(f'\nData types:')
print(data.dtypes.to_string())
print(f'\nMissing values:')
print(data.isnull().sum()[data.isnull().sum() > 0].to_string())

---
## 1 · Pandas Mastery: The Operations You'll Use Daily

In [ ]:
# Cell 2 — Essential EDA commands
import pandas as pd

# === THE 5 COMMANDS YOU RUN FIRST ON ANY DATASET ===

# 1. Shape & types
print(f'Shape: {data.shape}')
print(f'\n--- .info() ---')
data.info()

# 2. Statistical summary
print(f'\n--- .describe() ---')
print(data.describe().round(2).to_string())

# 3. Missing values
print(f'\n--- Missing Values ---')
missing = data.isnull().sum()
missing_pct = (missing / len(data) * 100).round(1)
print(pd.DataFrame({'count': missing[missing>0], 'percent': missing_pct[missing_pct>0]}).to_string())

# 4. Duplicates
print(f'\nDuplicate rows: {data.duplicated().sum()}')

# 5. Target distribution (for classification)
print(f'\nTarget distribution (churned):')
print(data['churned'].value_counts(normalize=True).round(3).to_string())
print(f'Class imbalance ratio: {data["churned"].value_counts()[0] / data["churned"].value_counts()[1]:.1f}:1')

---
## 2 · Visual EDA: Finding Problems Your Eyes Catch Faster Than Code

In [ ]:
# Cell 3 — Visual EDA
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('EDA Dashboard: Churn Prediction Dataset', fontsize=14, fontweight='bold')

# 1. Age distribution (spot outlier: age=350)
axes[0,0].hist(data['age'].dropna(), bins=40, color='steelblue', alpha=0.8)
axes[0,0].set_title('Age Distribution')
axes[0,0].set_xlabel('Age')
axes[0,0].annotate('Outlier!', xy=(350, 1), fontsize=10, color='red', fontweight='bold')

# 2. Income distribution (log-normal)
axes[0,1].hist(data['income'].dropna(), bins=40, color='coral', alpha=0.8)
axes[0,1].set_title('Income Distribution (log-normal)')

# 3. Target distribution
data['churned'].value_counts().plot.bar(ax=axes[0,2], color=['steelblue', 'coral'])
axes[0,2].set_title('Target: Churned (Class Imbalance!)')
axes[0,2].set_xticklabels(['Not Churned', 'Churned'], rotation=0)

# 4. Correlation heatmap
numeric_cols = data.select_dtypes(include=[np.number]).columns[:6]
sns.heatmap(data[numeric_cols].corr(), annot=True, fmt='.2f', cmap='RdBu_r', ax=axes[1,0])
axes[1,0].set_title('Feature Correlations')

# 5. Feature vs target (churn by country)
data.groupby('country')['churned'].mean().sort_values().plot.barh(ax=axes[1,1], color='steelblue')
axes[1,1].set_title('Churn Rate by Country')
axes[1,1].set_xlabel('Churn Rate')

# 6. Missing value pattern
missing_data = data.isnull().sum()[data.isnull().sum() > 0]
missing_data.plot.bar(ax=axes[1,2], color='coral')
axes[1,2].set_title('Missing Values by Column')
axes[1,2].set_ylabel('Count')

plt.tight_layout()
plt.savefig('eda_dashboard.png', dpi=100)
plt.show()

print('Findings from visual EDA:')
print('  1. Age has an outlier at 350 (impossible — data entry error)')
print('  2. Income is log-normal → may need log transform for models')
print('  3. Target is imbalanced (80/20) → need stratified splits, SMOTE, or class weights')
print('  4. income and credit_score have missing values → need imputation strategy')

---
## 3 · Data Cleaning Pipeline

In [ ]:
# Cell 4 — Production data cleaning pipeline
import pandas as pd
import numpy as np

def clean_dataset(df):
    """Production-grade data cleaning pipeline.
    
    Order matters! Clean in this sequence:
    1. Remove duplicates
    2. Fix outliers / impossible values
    3. Handle missing values
    4. Type conversions
    """
    df = df.copy()
    log = []
    
    # 1. Remove duplicates
    n_before = len(df)
    df = df.drop_duplicates()
    log.append(f'Removed {n_before - len(df)} duplicate rows')
    
    # 2. Fix outliers / impossible values
    # Age: must be 0-120
    invalid_age = (df['age'] < 0) | (df['age'] > 120)
    df.loc[invalid_age, 'age'] = np.nan  # Set to NaN for imputation
    log.append(f'Fixed {invalid_age.sum()} impossible age values')
    
    # Income: must be positive
    invalid_income = df['income'] < 0
    df.loc[invalid_income, 'income'] = np.nan
    log.append(f'Fixed {invalid_income.sum()} negative income values')
    
    # Credit score: must be 300-850
    invalid_cs = (df['credit_score'] < 300) | (df['credit_score'] > 850)
    df.loc[invalid_cs, 'credit_score'] = np.nan
    log.append(f'Fixed {invalid_cs.sum()} invalid credit scores')
    
    # 3. Handle missing values
    # Numeric: median (robust to outliers)
    for col in ['age', 'income', 'credit_score']:
        median_val = df[col].median()
        missing = df[col].isnull().sum()
        df[col].fillna(median_val, inplace=True)
        if missing > 0:
            log.append(f'Imputed {missing} missing {col} values with median ({median_val:.0f})')
    
    # Categorical: mode
    df['satisfaction_score'].fillna(df['satisfaction_score'].mode()[0], inplace=True)
    
    # 4. Type conversions
    df['customer_id'] = df['customer_id'].astype(str)
    df['churned'] = df['churned'].astype(int)
    
    # Report
    print('=== Cleaning Report ===')
    for entry in log:
        print(f'  ✓ {entry}')
    print(f'  Final: {len(df)} rows, {df.isnull().sum().sum()} remaining NaNs')
    
    return df

df_clean = clean_dataset(data)

---
## 4 · Feature Engineering for ML

In [ ]:
# Cell 5 — Feature transformations
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pandas as pd
import numpy as np

df = df_clean.copy()

# === Encoding categorical variables ===
# Option 1: One-hot encoding (for low cardinality: country has 6 values)
country_dummies = pd.get_dummies(df['country'], prefix='country')
print('One-hot encoding (country):')
print(f'  Before: 1 column (country)  →  After: {country_dummies.shape[1]} columns')
print(f'  Values: {df["country"].unique().tolist()}')

# === Scaling numeric features ===
# StandardScaler: mean=0, std=1 (required for: logistic regression, SVM, neural networks)
# NOT required for: tree-based models (XGBoost, Random Forest)
numeric_features = ['age', 'income', 'credit_score', 'account_age_months', 'num_products']

scaler = StandardScaler()
df_scaled = pd.DataFrame(
    scaler.fit_transform(df[numeric_features]),
    columns=[f'{c}_scaled' for c in numeric_features],
    index=df.index
)

print(f'\nScaling (StandardScaler):')
print(f'  Before: income range [{df["income"].min():.0f}, {df["income"].max():.0f}]')
print(f'  After:  income range [{df_scaled["income_scaled"].min():.2f}, {df_scaled["income_scaled"].max():.2f}]')

# === Log transform for skewed distributions ===
df['income_log'] = np.log1p(df['income'])  # log1p handles 0 values safely
print(f'\nLog transform (income):')
print(f'  Skewness before: {df["income"].skew():.2f}')
print(f'  Skewness after:  {df["income_log"].skew():.2f}  (closer to 0 = more normal)')

---
## 5 · Data Validation (Connects to NB07)

In production, data validation runs AUTOMATICALLY in your pipeline. Great Expectations (NB07) codifies your data assumptions as tests.

| Validation | What It Checks | Why |
|-----------|---------------|-----|
| Schema | Column names, types | Upstream schema changes break pipelines |
| Range | Min/max values | Outliers corrupt models |
| Null rate | % missing per column | Sudden null spikes = data source issue |
| Distribution | Statistical similarity to training | Drift = model degradation (NB35) |
| Uniqueness | No unexpected duplicates | ETL bugs |

In [ ]:
# Cell 6 — Simple data validation
def validate_dataset(df, rules):
    """Lightweight data validation (production: use Great Expectations, NB07)."""
    results = []
    for rule in rules:
        name = rule['name']
        passed = rule['check'](df)
        status = '✅ PASS' if passed else '❌ FAIL'
        results.append({'rule': name, 'status': status})
        print(f'  {status}: {name}')
    return results

rules = [
    {'name': 'No duplicate customer_ids', 'check': lambda df: df['customer_id'].nunique() == len(df)},
    {'name': 'Age between 0 and 120', 'check': lambda df: df['age'].between(0, 120).all()},
    {'name': 'Income is positive', 'check': lambda df: (df['income'] > 0).all()},
    {'name': 'Credit score 300-850', 'check': lambda df: df['credit_score'].between(300, 850).all()},
    {'name': 'No null values remaining', 'check': lambda df: df.isnull().sum().sum() == 0},
    {'name': 'Churn rate between 5-40%', 'check': lambda df: 0.05 <= df['churned'].mean() <= 0.40},
    {'name': 'At least 900 rows', 'check': lambda df: len(df) >= 900},
]

print('=== Data Validation Report (cleaned data) ===')
validate_dataset(df_clean, rules)

---
## ✅ Knowledge Check

### Q1: Imputation strategy
When should you use MEDIAN vs MEAN for imputing missing numeric values?

<details><summary>👉 Answer</summary>

Use **median** when data is skewed (income, house prices) — mean is pulled by outliers. Use **mean** when data is normally distributed (standardized test scores). In practice, median is the safer default. For production ML, use model-based imputation (KNN, iterative) from NB13 for better accuracy.
</details>

### Q2: Data leakage
You're predicting customer churn. Your model has 99% accuracy. You notice `last_interaction_date` is a feature. Is this leakage?

<details><summary>👉 Answer</summary>

Potentially YES. If `last_interaction_date` was populated AFTER the churn label was determined, it leaks future information. A churned customer's last interaction is by definition before they left. The model learns "no recent interactions = churned" which is a tautology, not a prediction. Remove temporal features that are set after the prediction point.
</details>

### Q3: When NOT to scale
You're using XGBoost for classification. Should you standardize your features?

<details><summary>👉 Answer</summary>

No. Tree-based models (XGBoost, Random Forest, LightGBM) are invariant to monotonic transformations. They split on "is feature > threshold?" which doesn't change with scaling. Scaling is only needed for distance-based (KNN, SVM) and gradient-based (neural networks, logistic regression) models.
</details>

---

## 🔨 Practical Practice

### Exercise 1: Missing Data Investigation
Check if the missing values in `income` are MCAR (Missing Completely at Random) or MNAR (Missing Not at Random). Hint: compare the churn rate for rows with vs without missing income. If they differ significantly, the missingness itself is informative.

### Exercise 2: Feature Interaction
Create 3 interaction features: `income_per_product`, `age_credit_ratio`, and `engagement_score` (combine `is_active`, `satisfaction_score`, `num_products`). Do any improve churn prediction?

### Exercise 3: Full Pipeline
Build a complete sklearn Pipeline that chains: imputation → scaling → encoding → model → evaluation. Use `ColumnTransformer` for heterogeneous feature types.

---

**Connections:** Data validation concepts → NB07 (Great Expectations) | Feature engineering → NB11 (Feast Feature Store) | Visual analysis → NB35 (EvidentlyAI Drift Detection)